# SimpleRepeatSeq — Notebook 01
## Carga del Dataset y Generación de Encodings

Carga el dataset FASTA, genera los 6 esquemas de codificación y los guarda como arrays `.npy`.

**Ejecutar este notebook antes de `02_experiments.ipynb`.**

### Estructura esperada
```
SimpleRepeatSeq/
├── data/
│   ├── SimpleRepeatSeq_dataset.fasta   ← coloca el dataset aquí
│   └── encoded/                         ← arrays .npy generados aquí
├── figures/
├── models/
└── notebooks/
    └── 01_load_data.ipynb  ← este notebook
```

In [ ]:
!pip install -q biopython numpy scikit-learn

In [ ]:
import os
import warnings
from collections import Counter
from itertools import product

import numpy as np
from Bio import SeqIO

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

# Rutas relativas al repo (el notebook vive en notebooks/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
FASTA_PATH = os.path.join(REPO_ROOT, 'data', 'SimpleRepeatSeq_dataset.fasta')
DATA_DIR   = os.path.join(REPO_ROOT, 'data', 'encoded')
FIG_DIR    = os.path.join(REPO_ROOT, 'figures')
MODEL_DIR  = os.path.join(REPO_ROOT, 'models')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'FASTA_PATH : {FASTA_PATH}')
print(f'DATA_DIR   : {DATA_DIR}')

---
## Carga del dataset

In [ ]:
seqs = list(SeqIO.parse(FASTA_PATH, 'fasta'))
print(f'Total de secuencias: {len(seqs)}')
print('Ejemplos:')
for s in seqs[:3]:
    print(f'  {s.id}  ({len(s.seq)} pb)')

In [ ]:
# Diagnóstico: caracteres y longitudes
todos = Counter()
for s in seqs:
    todos.update(str(s.seq).upper())

longitudes = [len(s.seq) for s in seqs]
print('Caracteres únicos:', sorted(todos.keys()))
print('Frecuencias:', dict(todos))
print(f'\nLongitud  min={min(longitudes)}  max={max(longitudes)}  '
      f'promedio={sum(longitudes)/len(longitudes):.1f}')

In [ ]:
MAX_LEN = max(len(s.seq) for s in seqs)
print(f'MAX_LEN para padding: {MAX_LEN}')

---
## Etiquetas

El formato de ID es `>CAG_huntington_007#simpleSeq#210`; la clase es el segundo campo.

In [ ]:
labels       = [s.id.split('#')[1] for s in seqs]
unique_labels = sorted(set(labels))
label_to_int  = {'no_simpleSeq': 0, 'simpleSeq': 1}
y_encoded     = np.array([label_to_int[lab] for lab in labels])

print('Etiquetas únicas:', unique_labels)
print('Mapeo:', label_to_int)
print(f'Shape y: {y_encoded.shape}')
print('Distribución:', dict(zip(*np.unique(y_encoded, return_counts=True))))

np.save(os.path.join(DATA_DIR, 'y_labels.npy'), y_encoded)
print('\ny_labels.npy guardado.')

---
## Funciones de encoding

Valores exactos de la especificación del proyecto.

| Esquema | Mapeo / descripción |
|---------|---------------------|
| DAX | A=2, C=0, G=3, T=1, N=1.5 |
| EIIP | A=0.126, C=0.134, G=0.081, T=0.134 |
| Complementario | A=2, C=-1, G=1, T=-2 (escala pu/pirimidina) |
| K-meros k=3/4/6 | Frecuencias normalizadas, Σ=ACGTN |

In [ ]:
def _pad_1d(vec, max_len, pad_value=0.0):
    n = len(vec)
    if n >= max_len:
        return vec[:max_len]
    return vec + [pad_value] * (max_len - n)


def dax_encode(sequence, max_len):
    mapping = {'A': 2.0, 'C': 0.0, 'G': 3.0, 'T': 1.0}
    return _pad_1d([mapping.get(nt, 1.5) for nt in sequence.upper()], max_len)


def eiip_encode(sequence, max_len):
    mapping = {'A': 0.126, 'C': 0.134, 'G': 0.081, 'T': 0.134}
    return _pad_1d([mapping.get(nt, 0.0) for nt in sequence.upper()], max_len)


def complementary_encode(sequence, max_len):
    mapping = {'A': 2.0, 'C': -1.0, 'G': 1.0, 'T': -2.0}
    return _pad_1d([mapping.get(nt, 0.0) for nt in sequence.upper()], max_len)


def _normalizar(sequence):
    return ''.join(nt if nt in 'ACGT' else 'N' for nt in sequence.upper())


def construir_kmer_counts(seqs_list, k, alfabeto='ACGTN'):
    """Frecuencias de k-meros normalizadas por ventana deslizante."""
    vocab = sorted(''.join(p) for p in product(alfabeto, repeat=k))
    kmer_to_idx = {km: i for i, km in enumerate(vocab)}
    V = len(vocab)
    X = np.zeros((len(seqs_list), V), dtype=np.float32)
    for i, s in enumerate(seqs_list):
        seq = _normalizar(str(s.seq))
        n_windows = len(seq) - k + 1
        if n_windows > 0:
            for j in range(n_windows):
                km = seq[j:j+k]
                if km in kmer_to_idx:
                    X[i, kmer_to_idx[km]] += 1
            X[i] /= n_windows
    return X


print('Funciones de encoding definidas.')

---
## Generación de todos los encodings

In [ ]:
# Encodings posicionales 1D (requieren padding a MAX_LEN)
for enc_name, enc_fn in [
    ('dax',           dax_encode),
    ('eiip',          eiip_encode),
    ('complementary', complementary_encode),
]:
    X = np.array(
        [enc_fn(str(s.seq), MAX_LEN) for s in seqs],
        dtype=np.float32
    )
    path = os.path.join(DATA_DIR, f'X_{enc_name}.npy')
    np.save(path, X)
    print(f'X_{enc_name}: {X.shape}  → {path}')

In [ ]:
# K-meros (k=3, 4, 6)
for k in [3, 4, 6]:
    Xk = construir_kmer_counts(seqs, k)
    path = os.path.join(DATA_DIR, f'X_kmer{k}.npy')
    np.save(path, Xk)
    print(f'X_kmer{k}: {Xk.shape}  (5^{k}={5**k} features)  → {path}')

In [ ]:
# Verificación final
archivos = ['X_dax', 'X_eiip', 'X_complementary', 'X_kmer3', 'X_kmer4', 'X_kmer6', 'y_labels']
print(f'{"Archivo":<25} {"Shape":<20} {"dtype":<12} {"MB":>8}')
print('-' * 70)
for a in archivos:
    path = os.path.join(DATA_DIR, f'{a}.npy')
    if os.path.exists(path):
        arr = np.load(path)
        mb  = os.path.getsize(path) / (1024 * 1024)
        print(f'{a+".npy":<25} {str(arr.shape):<20} {str(arr.dtype):<12} {mb:>8.2f}')
    else:
        print(f'{a+".npy":<25} FALTA')

print('\n✓ Encodings listos. Ejecuta 02_experiments.ipynb para los experimentos.')